In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable

## Functions

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

def clean_up_header(hdu):

    hdr_new = fits.Header()
    for card in hdu[0].header.cards:
        if card.keyword.strip() != "":
            hdr_new.append(card)

    return hdr_new

## Read in GMIMS linear fit RMs and clean up header

In [ ]:
hdu_G_RM = fits.open('/srv/aordog/cgps_gmims_data/RM_G.fits')
G_RM = hdu_G_RM[0].data
print(G_RM.shape)

hdr_G_RM = clean_up_header(hdu_G_RM)
del hdr_G_RM['CROTA1']
hdr_G_RM['WCSAXES'] = 2
print(repr(hdr_G_RM))

y_coords, x_coords = np.indices((hdr_G_RM['NAXIS2'], hdr_G_RM['NAXIS1']))
G_RM_coords = wcs.all_pix2world(x_coords, y_coords, 0)
G_RM_l = G_RM_coords[0][0]
G_RM_b = G_RM_coords[1][:,0]

del G_RM_coords; del y_coords; del x_coords
gc.collect()

## Read in GMIMS FD cube, clean up header and make 2D version

In [ ]:
hdu_G_FD = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
G_FD = hdu_G_FD[0].data
print(G_FD.shape)

hdr_G_FD = clean_up_header(hdu_G_FD)
del hdr_G_FD['WCSAXES']
del hdr_G_FD['LONPOLE']
del hdr_G_FD['LATPOLE']
del hdr_G_FD['ORIGIN']
del hdr_G_FD['INSTRUME']
del hdr_G_FD['COMMENT']
del hdr_G_FD['HISTORY']
print(repr(hdr_G_FD))

hdr_G_FD_2D = hdr_G_FD.copy()
hdr_G_FD_2D['NAXIS'] = 2
del hdr_G_FD_2D['NAXIS3']
del hdr_G_FD_2D['CRPIX3']
del hdr_G_FD_2D['CDELT3']
del hdr_G_FD_2D['CUNIT3']
del hdr_G_FD_2D['CTYPE3']
del hdr_G_FD_2D['CRVAL3']
print('---------------------')
print(repr(hdr_G_FD_2D))

hdr_G_FD_2Dfix = hdr_G_FD_2D.copy()
hdr_G_FD_2Dfix['CRPIX1'] = 1
hdr_G_FD_2Dfix['CRPIX2'] = 1
hdr_G_FD_2Dfix['CRVAL1'] = 360
hdr_G_FD_2Dfix['CRVAL2'] = -90
#hdr_G_FD_2Dfix[''] = 
#hdr_G_FD_2Dfix[''] =
print('---------------------')
print(repr(hdr_G_FD_2Dfix))

## Calculate GMIMS full moments and peaks

In [ ]:
FD_idx = np.linspace(0,hdr_G_FD['NAXIS3']-1,hdr_G_FD['NAXIS3'])
FD_ax = hdr_G_FD['CRVAL3']+(FD_idx - hdr_G_FD['CRPIX3']+1)*hdr_G_FD['CDELT3']
print(FD_ax.shape)

G_moments = FD_moments(G_FD,0.03,400,FD_ax)
G_M1 = G_moments['M1']

peak_FD = np.empty_like(G_FD[0,:,:])
print(peak_FD.shape)
for i in range(0,peak_FD.shape[0]):
    for j in range(0,peak_FD.shape[1]):
        idx = np.where(G_FD[:,i,j] == np.nanmax(G_FD[:,i,j]))[0][0]
        peak_FD[i,j] = FD_ax[idx]

## Check maps

In [ ]:
fig = plt.figure(figsize=(20, 9))

wcs = WCS(hdr_G_RM)
print(wcs)
c = SkyCoord([120,60], [-5,5], frame=Galactic, unit="deg")

ax = fig.add_axes(311, projection=wcs)
ax.imshow(G_RM, vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
ax.set_xlim(wcs.world_to_pixel(c)[0])
ax.set_ylim(wcs.world_to_pixel(c)[1])

plt.show()

In [ ]:
peak_FD_reproj, footprint = reproject_interp((peak_FD,WCS(hdr_G_FD_2D)), WCS(hdr_G_FD_2Dfix))

fig = plt.figure(figsize=(20, 9))

wcs = WCS(hdr_G_FD_2Dfix)
print(wcs)
c = SkyCoord([192,60], [-5,5], frame=Galactic, unit="deg")

ax = fig.add_axes(311, projection=wcs.celestial)
ax.imshow(peak_FD_reproj, vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
ax.set_xlim(wcs.world_to_pixel(c)[0])
ax.set_ylim(wcs.world_to_pixel(c)[1])

plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 9))

c = SkyCoord([192,50], [-5,7], frame=Galactic, unit="deg")

lon_use = c.l.wrap_at(180*u.deg).deg
lat_use = c.b.deg

c2 = SkyCoord(lon_use, lat_use, frame=Galactic, unit="deg")

wcs = WCS(hdr_G_RM)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0]) ; i2 = int(x[1]) ; j1 = int(y[0]) ; j2 = int(y[1])
plt.subplot(311,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude') ; plt.ylabel('Galactic Latitude')

wcs = WCS(hdr_G_FD_2D)
print(wcs)
x, y = np.round(wcs.world_to_pixel(c2))
i1 = int(x[0]) ; i2 = int(x[1]) ; j1 = int(y[0]) ; j2 = int(y[1])
print(i1,i2,j1,j2)
plt.subplot(312,projection=wcs[j1:j2,i1:i2])
plt.imshow(peak_FD[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude') ; plt.ylabel('Galactic Latitude')


In [ ]:
fs = 18
fig = plt.figure(figsize=(20, 5))

c = SkyCoord([192,50], [-5,7], frame=Galactic, unit="deg")

wcs = WCS(hdu_G_RM[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(211,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.contour(G_RM[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

#plt.subplot(212,projection=wcs[j1:j2,i1:i2])
#plt.imshow(peak_FD_reproj[j1:j2,i1:i2], vmin=-50, vmax=50, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude')
#plt.ylabel('Galactic Latitude')

In [ ]:
print(FD_ax[60:141])
print(FD_ax[140])

In [ ]:
FD_interp = np.empty([len(FD_ax[60:141]),peak_FD_reproj.shape[0],peak_FD_reproj.shape[1]])
print(FD_interp.shape)

for i in range(60,141):
    print(i,FD_ax[i])
    FD_interp[i,:,:], footprint = reproject_interp((G_FD[i,:,:],hdrFD_2D), hdu_G_RM[0].header)